# 🦥 1-Click Fast Fine-Tuning & GGUF Export with Unsloth

Train a custom coding/chat model on free Google Colab GPU (T4/A100) and export directly to **GGUF for Ollama**.

### Step 1: Install Unsloth

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

### Step 2: Load Base Model (Qwen2.5-Coder-7B or Llama-3.2-3B)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

### Step 3: Format Dataset & Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# Upload your dataset_alpaca.json to Colab
dataset = load_dataset("json", data_files="dataset_alpaca.json", split="train")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "output",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)
trainer.train()

### Step 4: Export to GGUF (for Ollama)

In [ ]:
# Export to Q4_K_M GGUF format
model.save_pretrained_gguf("model_q4_k_m", tokenizer, quantization_method = "q4_k_m")
print("GGUF Export Complete! Download model_q4_k_m-unsloth.Q4_K_M.gguf and load into Ollama.")